In [ ]:
### langchain:
# It is an opensource framework used to build applications powered by LLM.
# Open AT GPT models
# Google gemini
# Anthrophic
# local Models as well

# prompt --> LLM --> Response

In [ ]:
!pip install langchain
!pip install langchain-community

In [ ]:
from google import genai
from google.colab import userdata
apk = userdata.get('MY_API_KEY3')
clinet = genai.Client(api_key= apk)

In [ ]:
from langchain_core import documents
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=0)
# texts = text_splitter.split_text(documents)

In [ ]:
!pip install yfinance

In [ ]:
import pandas as pd
import numpy as np
from google import genai
!pip install faiss-cpu
import faiss

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
import yfinance as yf

In [ ]:
ticker = "AAPL"
stock =yf.Ticker(ticker)


In [ ]:
historical_data = stock.history(period="6mo")
print(historical_data.tail())

In [ ]:
company_info = stock.info
print(company_info)

In [ ]:
news = stock.news
print(news)

In [ ]:
for item in news[:5]:
  print(item["content"]["title"])
  print(item["content"]["summary"])
  print("---"*50)

In [ ]:
## Create a Financial Knowledge Base
## Create Historical data + Company Info + News

In [ ]:
documents =[]
# 1. Company_info
company_text = f"""
Industry: {company_info["industry"]},
Sector; {company_info["sector"]},
Description: {company_info["longBusinessSummary"]}
"""
documents.append(company_text)

# 2. Current Price
price_text = f"""
Current Stock Price of {company_info.get('shortName', ticker)}:
${company_info.get('currentPrice', 'N/A')}
Previous Close: ${company_info.get('previousClose', 'N/A')}
52 Week High: ${company_info.get('fiftyTwoWeekHigh', 'N/A')}
52 Week Low: ${company_info.get('fiftyTwoWeekLow', 'N/A')}
Market Cap: ${company_info.get('marketCap', 'N/A')}
"""
documents.append(price_text)

# 3. News
news_texts = []
for item in news[:5]:
    news_texts.append(item["content"]["title"])
    news_texts.append(item["content"]["summary"])
news_text = "\n".join(news_texts)
documents.append(news_text)

In [ ]:
print(f"Total documents: {len(documents)}")
for i, d in enumerate(documents):
    print(f"\n--- Doc {i} ---\n{d[:200]}")

In [ ]:
# documents=[]
# news_text = f"""
# for item in news[:5]:
#   print(item["content"]["title"])
#   print(item["content"]["summary"])
#   """
# documents.append(news_text)

# print(documents)


In [ ]:
### 4. Historical Data
historical_text = historical_data.tail(30).to_string()
documents.append(historical_text)
print(documents)
# 4. Historical data
# documents.append(historical_data.tail(30).to_string())

In [ ]:
documents[0]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=0)
chunks = text_splitter.create_documents(documents)
print(len(chunks))
print(chunks)

In [ ]:
print(chunks[10])

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
text  = [doc.page_content for doc in chunks]
embeddings = embedding_model.encode(text)
print(embeddings)

In [ ]:
## Store that in Vector Database - FAISS

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

In [ ]:
from decorator import append
### Creating the Retrieval Function to Retrieve Data from vector DB, which will act as a User context for LLM

def retrieve_context(query,top_k=3):
  query_embedding = embedding_model.encode([query])
  distances, indices = index.search(np.array(query_embedding), k = top_k)

  retrieved_docs =[]
  for idx in indices[0]:
    retrieved_docs.append(chunks[idx].page_content)

  return "\n\n".join(retrieved_docs)

In [ ]:
## Conversational Bot (Agumentation - RAG based Chatbot)
from google.colab import userdata
apk = userdata.get('MY_API_KEY3')
client = genai.Client(api_key = apk)

In [ ]:
chat_history = []
def ask_question(question):
  global chat_history
  context = retrieve_context(question,top_k=3)
  history = "\n".join(chat_history)

  prompt = f"""
You are an AI Financial Assistant specialized in stock market analysis.

Use the provided context to answer the user's question accurately.

Conversation History:
{history}

Context:
{context}

User Question:
{question}

Instructions:
- Answer only using the provided context
- If the answer is not available in the context, say:
- I could not find enough information in the available financial data.
- Summarize financial news clearly
- Explain stock trends in simple language
- Be concise but informative
- Use a professional and conversational tone
- Mention important market insights when relevant
- Do not make up financial information
"""

  response = client.models.generate_content(
      model = "gemini-2.5-flash",
      contents = prompt
  )
  answer = response.text
  chat_history.append(f"Question: ,{question}")
  chat_history.append(f"Answer: ,{answer}")


  return answer

In [ ]:
response = ask_question(
    "What is the stock price of Apple Inc?"
)
print(response)

In [ ]:
response = ask_question(
    "Provide me the Industry and Technolgy?"
)
print(response)

In [ ]:
import pickle

In [111]:
with open("faiss_index.pkl", "wb") as f:
  pickle.dump(index, f)
with open("documents.pkl", "wb") as f:
  pickle.dump(documents, f)
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
print("faiss_index.pkl , documents.pkl and chunks.pkl saved!")

faiss_index.pkl , documents.pkl and chunks.pkl saved!


In [ ]:
# Check this in a cell:
import pickle
with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

print(f"Number of documents: {len(documents)}")
print(documents[0] if documents else "EMPTY!")

In [ ]:
import yfinance as yf
ticker = yf.Ticker("AAPL")
info = ticker.info
print(info.get("currentPrice"))   # should print a number
print(info.get("shortName"))

In [ ]:
# Check what the search actually returns:
import pickle
with open("faiss_index.pkl", "rb") as f:
    index = pickle.load(f)

# If using sentence-transformers:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
query_vec = model.encode(["What is the stock price of Apple Inc?"])

D, I = index.search(query_vec, k=3)
print("Distances:", D)
print("Indices:", I)

In [ ]:
!pip install streamlit


In [ ]:
import streamlit